In [14]:
from langgraph.graph import StateGraph, START, END
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import TypedDict
from dotenv import load_dotenv


In [15]:
load_dotenv()  # Load environment variables from .env file

MODEL_NAME = "gemini-3.8-flash"
precise_model = ChatGoogleGenerativeAI(
    model=MODEL_NAME,
    temperature=0,
    max_output_tokens=512,
    thinking_budget=0,
)

In [16]:
class BlogState(TypedDict):
    topic: str
    outline: str
    content: str

In [17]:
def create_outline(state: BlogState) -> BlogState:
    prompt = f"Create a detailed outline for a blog post on the topic: '{state['topic']}'."
    outline = precise_model.invoke(prompt).content
    state['outline'] = outline
    return state

In [18]:
def create_blog(state: BlogState) -> BlogState:
    title = state["topic"]
    outline = state["outline"]

    prompt = f"""
Write a blog post titled "{title}" based on the following outline:

{outline}
"""

    response = precise_model.invoke(prompt)

    state["content"] = response.content

    return state

In [19]:
graph=StateGraph(BlogState)

# node
graph.add_node('create_outline',create_outline)
graph.add_node('create_blog',create_blog)

# edge
graph.add_edge(START,'create_outline')
graph.add_edge('create_outline','create_blog')
graph.add_edge('create_blog',END)

workflow= graph.compile()


In [ ]:
initial_state = {"topic": "The Future of Artificial Intelligence"}
final_state = workflow.invoke(initial_state)
print(final_state["content"])


Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.
